# Desafio Mezzalira · Etapa 1: do dado bagunçado ao dado útil

**Metalúrgica Mezzalira · Processo seletivo do Citi (área de dados)**

Este notebook refaz em Python a limpeza feita no Google Sheets (arquitetura **Bronze → Prata → Log → Ouro**) e confere se o resultado bate com as abas da planilha.

**Pergunta:** qual cidade mais comprou da Metalúrgica e em qual mês?

**Como usar:** `Ambiente de execução → Executar tudo`. O notebook lê a planilha direto do Google Sheets (acesso público de leitura). Se não conseguir, pede o upload do arquivo `.xlsx`.


In [1]:
import io
import re
import urllib.request
from datetime import datetime

import pandas as pd

def brl(valor):
    """1234.5 -> 'R$ 1.234,50' (formato brasileiro)."""
    return "R$ " + f"{valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


pd.options.display.float_format = brl

SHEET_ID = "1XKk1FASJ2rDJvFm9eHV_3j7j_0KjMhtWl9IYu-xO_io"
URL_XLSX = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=xlsx"
ARQUIVO_LOCAL = None  # use só para testar fora do Colab
COLUNAS = ["Pedido", "Data", "Cliente", "Cidade", "Produto", "Valor"]
MESES_PT = {"jan": 1, "fev": 2, "mar": 3, "abr": 4, "mai": 5, "jun": 6,
            "jul": 7, "ago": 8, "set": 9, "out": 10, "nov": 11, "dez": 12}
CIDADES_RECIFE = {"recife", "rec"}   # decisão com dúvida: REC = Recife (ver Log)


## 1. Bronze: leitura do dado original
A aba Bronze é a cópia intocada da planilha original. Aqui ela só é lida, nunca alterada.

In [2]:
def abrir_planilha():
    """Tenta ler direto do Google Sheets; se falhar, pede o upload do .xlsx."""
    try:
        dados = urllib.request.urlopen(URL_XLSX, timeout=30).read()
        print("Planilha lida direto do Google Sheets.")
        return io.BytesIO(dados)
    except Exception as erro:
        print(f"Leitura direta não funcionou ({type(erro).__name__}).")
    if ARQUIVO_LOCAL:
        return ARQUIVO_LOCAL
    from google.colab import files
    print("Envie o arquivo .xlsx da planilha de pedidos.")
    enviado = files.upload()
    nome = next(iter(enviado))
    return io.BytesIO(enviado[nome])


planilha = pd.ExcelFile(abrir_planilha())
abas = planilha.sheet_names
print("Abas encontradas:", abas)

aba_bronze = "Bronze" if "Bronze" in abas else abas[0]
bruto = planilha.parse(aba_bronze).iloc[:, :6]
bruto.columns = COLUNAS

# remove só as linhas vazias do FIM da planilha (as vazias do meio precisam ser contadas)
ultimo = bruto["Pedido"].last_valid_index()
bruto = bruto.loc[:ultimo].copy()
bruto["Linha_Bronze"] = bruto.index + 2   # linha correspondente na aba Bronze
print(f"Linhas de dados na Bronze: {len(bruto)}")
bruto.head()


Planilha lida direto do Google Sheets.
Abas encontradas: ['Bronze', 'Prata', 'Ouro', 'Log']
Linhas de dados na Bronze: 60


,Pedido,Data,Cliente,Cidade,Produto,Valor,Linha_Bronze
0,"R$ 1.001,00",2026-01-01,Irrigação Vale Forte,Petrolina,"Tubo galvanizado 2""",3260,2
1,"R$ 1.002,00",2026-01-02 00:00:00,Grades Caruaru,Caruaru,"Barra chata 1""",3570,3
2,"R$ 1.003,00",2026-01-06 00:00:00,Serralheria Capibaribe,recife,Telha trapezoidal,1940,4
3,"R$ 1.004,00",2026-01-07 00:00:00,Serralheria Capibaribe,Recife,Chapa de aço 3 mm,4910,5
4,"R$ 1.005,00",2026-01-16,Agro Estruturas São Francisco,Petrolina,Perfil U enrijecido,4500,6


## 2. Diagnóstico: os cinco problemas plantados

In [3]:
def tipo_data(valor):
    if isinstance(valor, (datetime, pd.Timestamp)):
        return "data real"
    texto = str(valor).strip()
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", texto):
        return "texto aaaa-mm-dd"
    if re.fullmatch(r"\d{1,2}-[a-zç]{3}-\d{4}", texto.lower()):
        return "texto dd-mmm-aaaa"
    return "formato desconhecido"


vazias = bruto[bruto["Pedido"].isna()]
dados = bruto.dropna(subset=["Pedido"]).copy()

duplicadas = dados[dados.duplicated(subset=COLUNAS, keep="first")]
unicos = dados.drop(duplicadas.index)   # os demais problemas são contados nos pedidos que ficam

datas_texto = unicos[unicos["Data"].apply(tipo_data) != "data real"]
valores_texto = unicos[unicos["Valor"].apply(lambda v: isinstance(v, str))]
cidades_fora = unicos[unicos["Cidade"].str.strip().str.lower().isin(CIDADES_RECIFE)
                      & (unicos["Cidade"].str.strip() != "Recife")]

print("1) Linhas vazias:", len(vazias), "→ linhas", vazias["Linha_Bronze"].tolist())
print("2) Pedidos duplicados:", len(duplicadas), "→ linhas", duplicadas["Linha_Bronze"].tolist())
print("3) Cidade fora do padrão:", len(cidades_fora), "→", cidades_fora["Cidade"].value_counts().to_dict())
print("4) Valores digitados como texto:", len(valores_texto), "→ linhas", valores_texto["Linha_Bronze"].tolist())
print("5) Datas em texto:", len(datas_texto))
print(datas_texto["Data"].apply(tipo_data).value_counts().to_string())


1) Linhas vazias: 3 → linhas [16, 35, 53]
2) Pedidos duplicados: 4 → linhas [25, 33, 36, 61]
3) Cidade fora do padrão: 10 → {'recife': 5, 'REC': 5}
4) Valores digitados como texto: 7 → linhas [15, 21, 28, 29, 32, 44, 51]
5) Datas em texto: 17
Data
texto aaaa-mm-dd     11
texto dd-mmm-aaaa     6


## 3. Prata: correção dos problemas
Cada função trata um problema. A coluna `Linha_Bronze` é mantida para rastrear a origem de cada pedido.

In [4]:
def converter_data(valor):
    """Aceita data real, 'aaaa-mm-dd' e 'dd-mmm-aaaa' (mês abreviado em português)."""
    if isinstance(valor, (datetime, pd.Timestamp)):
        return pd.Timestamp(valor).normalize()
    texto = str(valor).strip()
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", texto):
        return pd.Timestamp(texto)
    achado = re.fullmatch(r"(\d{1,2})-([a-zç]{3})-(\d{4})", texto.lower())
    if achado:
        return pd.Timestamp(int(achado[3]), MESES_PT[achado[2]], int(achado[1]))
    raise ValueError(f"Formato de data não reconhecido: {valor!r}")


def converter_valor(valor):
    """'R$ 1.200,00' → 1200.0 ; números passam direto."""
    if isinstance(valor, str):
        limpo = valor.replace("R$", "").replace(".", "").replace(",", ".").strip()
        return float(limpo)
    return float(valor)


def padronizar_cidade(cidade):
    texto = str(cidade).strip()
    return "Recife" if texto.lower() in CIDADES_RECIFE else texto


prata = dados.copy()                                  # 1) linhas vazias já ficaram de fora
prata["Data"] = prata["Data"].apply(converter_data)   # 5) datas
prata["Valor"] = prata["Valor"].apply(converter_valor)    # 4) valores
prata["Cidade"] = prata["Cidade"].apply(padronizar_cidade)  # 3) cidades
prata["Pedido"] = prata["Pedido"].astype(int)

# 2) duplicados: compara todas as colunas (sem Linha_Bronze) e mantém a 1ª ocorrência
prata = prata.drop_duplicates(subset=COLUNAS, keep="first").reset_index(drop=True)

print(f"Pedidos na Prata: {len(prata)} | pedidos únicos: {prata['Pedido'].nunique()}")
prata.head()


Pedidos na Prata: 53 | pedidos únicos: 53


,Pedido,Data,Cliente,Cidade,Produto,Valor,Linha_Bronze
0,1001,2026-01-01,Irrigação Vale Forte,Petrolina,"Tubo galvanizado 2""","R$ 3.260,00",2
1,1002,2026-01-02,Grades Caruaru,Caruaru,"Barra chata 1""","R$ 3.570,00",3
2,1003,2026-01-06,Serralheria Capibaribe,Recife,Telha trapezoidal,"R$ 1.940,00",4
3,1004,2026-01-07,Serralheria Capibaribe,Recife,Chapa de aço 3 mm,"R$ 4.910,00",5
4,1005,2026-01-16,Agro Estruturas São Francisco,Petrolina,Perfil U enrijecido,"R$ 4.500,00",6


## 4. Log: registro das correções
Gerado automaticamente a partir do diagnóstico, com as linhas afetadas na Bronze.

In [5]:
def linhas(df):
    return ", ".join(str(n) for n in df["Linha_Bronze"])

recife_min = unicos[unicos["Cidade"].str.strip() == "recife"]
rec_sigla = unicos[unicos["Cidade"].str.strip() == "REC"]
datas_iso = datas_texto[datas_texto["Data"].apply(tipo_data) == "texto aaaa-mm-dd"]
datas_mmm = datas_texto[datas_texto["Data"].apply(tipo_data) == "texto dd-mmm-aaaa"]

log = pd.DataFrame([
    ("Linhas totalmente vazias", "Removidas da Prata", linhas(vazias), "Sem dado, não representam pedido."),
    ("Pedidos duplicados (idênticos em todas as colunas)", "Removida a 2ª ocorrência, mantida a 1ª", linhas(duplicadas),
     "Linhas idênticas: a escolha não altera valores. Sem remover, Caruaru passaria Recife."),
    ('Cidade "recife" em minúsculas', 'Padronizada para "Recife"', linhas(recife_min), "Mesma cidade, só mudava a grafia."),
    ('Cidade "REC"', 'Padronizada para "Recife"', linhas(rec_sigla),
     "DÚVIDA: assumi que REC é sigla de Recife; nenhuma outra cidade da base começa com REC."),
    ("Datas em texto aaaa-mm-dd", "Convertidas para data real", linhas(datas_iso), "Formato sem ambiguidade entre dia e mês."),
    ("Datas em texto dd-mmm-aaaa (mês em português)", "Convertidas para data real", linhas(datas_mmm),
     "O parser lê a abreviação do mês (fev, mar...) de forma explícita."),
    ("Valores digitados como texto (ex.: R$ 630,00)", "Convertidos para número", linhas(valores_texto),
     "Sem isso a soma ignora esses pedidos."),
], columns=["O que encontrei", "O que fiz", "Linhas afetadas (Bronze)", "Justificativa"])

pd.set_option("display.max_colwidth", None)
log


,O que encontrei,O que fiz,Linhas afetadas (Bronze),Justificativa
0,Linhas totalmente vazias,Removidas da Prata,"16, 35, 53","Sem dado, não representam pedido."
1,Pedidos duplicados (idênticos em todas as colunas),"Removida a 2ª ocorrência, mantida a 1ª","25, 33, 36, 61","Linhas idênticas: a escolha não altera valores. Sem remover, Caruaru passaria Recife."
2,"Cidade ""recife"" em minúsculas","Padronizada para ""Recife""","4, 21, 37, 39, 51","Mesma cidade, só mudava a grafia."
3,"Cidade ""REC""","Padronizada para ""Recife""","9, 18, 28, 32, 59",DÚVIDA: assumi que REC é sigla de Recife; nenhuma outra cidade da base começa com REC.
4,Datas em texto aaaa-mm-dd,Convertidas para data real,"2, 6, 7, 17, 21, 23, 24, 37, 38, 47, 56",Formato sem ambiguidade entre dia e mês.
5,Datas em texto dd-mmm-aaaa (mês em português),Convertidas para data real,"8, 28, 32, 40, 43, 45","O parser lê a abreviação do mês (fev, mar...) de forma explícita."
6,"Valores digitados como texto (ex.: R$ 630,00)",Convertidos para número,"15, 21, 28, 29, 32, 44, 51",Sem isso a soma ignora esses pedidos.


## 5. Ouro: faturamento por cidade e por mês

In [6]:
prata["Mes"] = prata["Data"].dt.to_period("M").astype(str)

ouro = prata.pivot_table(index="Cidade", columns="Mes", values="Valor",
                         aggfunc="sum", margins=True, margins_name="Total", fill_value=0)
ouro


Mes,2026-01,2026-02,2026-03,2026-04,Total
Cidade,,,,,
Caruaru,"R$ 3.570,00","R$ 15.030,00","R$ 17.610,00","R$ 6.930,00","R$ 43.140,00"
Jaboatão dos Guararapes,"R$ 0,00","R$ 3.920,00","R$ 22.820,00","R$ 1.120,00","R$ 27.860,00"
Olinda,"R$ 0,00","R$ 10.100,00","R$ 7.710,00","R$ 11.190,00","R$ 29.000,00"
Petrolina,"R$ 9.110,00","R$ 8.770,00","R$ 1.590,00","R$ 3.610,00","R$ 23.080,00"
Recife,"R$ 6.850,00","R$ 16.240,00","R$ 29.730,00","R$ 2.440,00","R$ 55.260,00"
Total,"R$ 19.530,00","R$ 54.060,00","R$ 79.460,00","R$ 25.290,00","R$ 178.340,00"


## 6. Resposta

In [7]:
por_cidade = prata.groupby("Cidade")["Valor"].sum().sort_values(ascending=False)
cidade_lider = por_cidade.index[0]
por_mes = prata[prata["Cidade"] == cidade_lider].groupby("Mes")["Valor"].sum()
mes_pico = por_mes.idxmax()
total_mes = prata.groupby("Mes")["Valor"].sum()

print(f"Cidade que mais comprou: {cidade_lider} ({brl(por_cidade.iloc[0])} no período)")
print(f"Mês de maior faturamento dela: {mes_pico} ({brl(por_mes.max())})")
print(f"Mês de maior faturamento geral: {total_mes.idxmax()} ({brl(total_mes.max())})")
print(f"Período dos dados: {prata['Data'].min():%d/%m/%Y} a {prata['Data'].max():%d/%m/%Y}")


Cidade que mais comprou: Recife (R$ 55.260,00 no período)
Mês de maior faturamento dela: 2026-03 (R$ 29.730,00)
Mês de maior faturamento geral: 2026-03 (R$ 79.460,00)
Período dos dados: 01/01/2026 a 26/04/2026


### Por que a limpeza importa
Se os duplicados não fossem removidos, o ranking mudaria:

In [8]:
sem_remover = dados.copy()
sem_remover["Valor"] = sem_remover["Valor"].apply(converter_valor)
sem_remover["Cidade"] = sem_remover["Cidade"].apply(padronizar_cidade)

comparacao = pd.DataFrame({
    "Com a limpeza": por_cidade,
    "Sem remover duplicados": sem_remover.groupby("Cidade")["Valor"].sum(),
}).sort_values("Sem remover duplicados", ascending=False)
comparacao


,Com a limpeza,Sem remover duplicados
Cidade,,
Caruaru,"R$ 43.140,00","R$ 60.590,00"
Recife,"R$ 55.260,00","R$ 55.260,00"
Olinda,"R$ 29.000,00","R$ 29.000,00"
Jaboatão dos Guararapes,"R$ 27.860,00","R$ 27.860,00"
Petrolina,"R$ 23.080,00","R$ 23.080,00"


## 7. Validação
Conferências automáticas do resultado e, quando a planilha está acessível, comparação linha a linha com a aba Prata do Google Sheets.

In [9]:
ESPERADO = {
    "linhas vazias": 3,
    "duplicados": 4,
    "pedidos únicos": 53,
    "total geral": 178340.00,
    "cidade líder": "Recife",
    "total da cidade líder": 55260.00,
    "mês de pico": "2026-03",
    "valor do pico": 29730.00,
}
obtido = {
    "linhas vazias": len(vazias),
    "duplicados": len(duplicadas),
    "pedidos únicos": prata["Pedido"].nunique(),
    "total geral": round(prata["Valor"].sum(), 2),
    "cidade líder": cidade_lider,
    "total da cidade líder": round(por_cidade.iloc[0], 2),
    "mês de pico": mes_pico,
    "valor do pico": round(por_mes.max(), 2),
}
tudo_ok = True
for nome, esperado in ESPERADO.items():
    ok = obtido[nome] == esperado
    tudo_ok &= ok
    print(f"[{'OK' if ok else 'ERRO'}] {nome}: obtido={obtido[nome]} esperado={esperado}")

if "Prata" in abas:
    prata_sheets = planilha.parse("Prata").dropna(subset=["Pedido"])
    prata_sheets = prata_sheets.rename(columns={"Valor (R$)": "Valor"})
    junto = prata.merge(prata_sheets, on="Pedido", suffixes=("_py", "_sheets"))
    iguais = (
        (len(junto) == len(prata) == len(prata_sheets))
        and (pd.to_datetime(junto["Data_py"]) == pd.to_datetime(junto["Data_sheets"])).all()
        and (junto["Cidade_py"] == junto["Cidade_sheets"]).all()
        and ((junto["Valor_py"] - junto["Valor_sheets"].astype(float)).abs() < 0.005).all()
    )
    tudo_ok &= bool(iguais)
    print(f"[{'OK' if iguais else 'ERRO'}] Python = aba Prata do Sheets ({len(junto)} pedidos comparados)")
else:
    print("Aba Prata não disponível: comparação com o Sheets pulada.")

print()
print("TUDO CONFERE." if tudo_ok else "HÁ DIVERGÊNCIAS: revisar.")


[OK] linhas vazias: obtido=3 esperado=3
[OK] duplicados: obtido=4 esperado=4
[OK] pedidos únicos: obtido=53 esperado=53
[OK] total geral: obtido=178340.0 esperado=178340.0
[OK] cidade líder: obtido=Recife esperado=Recife
[OK] total da cidade líder: obtido=55260.0 esperado=55260.0
[OK] mês de pico: obtido=2026-03 esperado=2026-03
[OK] valor do pico: obtido=29730.0 esperado=29730.0
[OK] Python = aba Prata do Sheets (53 pedidos comparados)

TUDO CONFERE.


## Resumo

> **Recife** foi a cidade que mais comprou (R$ 55.260,00 no período), e **março de 2026** foi o seu mês de maior faturamento (R$ 29.730,00).

Decisões com dúvida (também na aba Log da planilha): remoção de 4 pedidos duplicados e interpretação de "REC" como Recife. Os dados vão de 01/01 a 26/04/2026, então abril está incompleto.
